# Parametric vs Nonparametric Models

**Topic:** Assumptions vs Flexibility

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, ToggleButtons, Output, HBox, VBox, Label
from IPython.display import display, clear_output
from scipy import stats
from sklearn.datasets import make_classification, make_regression, make_circles, make_moons
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the difference between parametric and nonparametric models in plain English
- **Explain** why nonparametric models need more data to perform well
- **Interpret** what happens to each model type when the parametric assumption is violated

> **Tip:** Start with **Assumptions met** (linear data) and a small sample size (n=10). Both models struggle. Increase n and watch which model recovers faster. Then switch to **Assumptions violated** and observe the parametric model plateau while KNN keeps improving.

---
## How we got here

In **13_interpretability_vs_complexity.ipynb** we placed algorithms on the interpretability-complexity spectrum. Now we cut across that spectrum differently: by asking whether the algorithm assumes a fixed shape for the data.

This distinction is more fundamental than it first appears. It determines how much data you need, how the model behaves when your assumptions are wrong, and how the model scales to new data.

---
## Why this matters for data science

Parametric models are efficient: if the assumption is correct, they extract maximum signal from limited data. They fail gracefully: a linear regression on nonlinear data at least fits the best possible line, which is often useful as a baseline.

Nonparametric models are flexible: they can fit any shape. But they are data hungry. A KNN with 10 training examples in 20 dimensions is choosing neighbors essentially at random.

The right choice depends on what you know about your data's structure and how much data you have.

---
## Try it yourself

In [2]:
np.random.seed(42)

out = Output()

n_slider = IntSlider(value=80, min=10, max=500, step=10,
    description="Sample size:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="420px"))

assumption_toggle = ToggleButtons(
    options=["Assumptions met (linear data)", "Assumptions violated (nonlinear data)"],
    value="Assumptions met (linear data)",
    description="", layout=widgets.Layout(width="460px"))

def render(n, violated):
    np.random.seed(42)
    if violated:
        X_raw  = np.linspace(0, 4*np.pi, n)
        y_true = np.sin(X_raw)
        y_data = y_true + np.random.normal(0, 0.3, n)
    else:
        X_raw  = np.linspace(0, 10, n)
        y_true = 2*X_raw
        y_data = y_true + np.random.normal(0, 2, n)

    X_2d = X_raw.reshape(-1, 1)
    lr   = LinearRegression().fit(X_2d, y_data)

    from sklearn.neighbors import KNeighborsRegressor
    knn_reg = KNeighborsRegressor(n_neighbors=min(5, max(1, n//8))).fit(X_2d, y_data)

    x_plot = np.linspace(X_raw.min(), X_raw.max(), 300).reshape(-1, 1)
    lr_mse  = mean_squared_error(y_data, lr.predict(X_2d))
    knn_mse = mean_squared_error(y_data, knn_reg.predict(X_2d))

    traces = [
        go.Scatter(x=X_raw, y=y_data, mode="markers",
            marker=dict(color=PALETTE["muted"], size=6, opacity=0.6), name="Data"),
        go.Scatter(x=x_plot.ravel(), y=lr.predict(x_plot), mode="lines",
            line=dict(color=PALETTE["primary"], width=2.5),
            name=f"Parametric (Linear) MSE={lr_mse:.2f}"),
        go.Scatter(x=x_plot.ravel(), y=knn_reg.predict(x_plot), mode="lines",
            line=dict(color=PALETTE["secondary"], width=2.5),
            name=f"Nonparametric (KNN) MSE={knn_mse:.2f}"),
    ]
    data_type = "Nonlinear" if violated else "Linear"
    layout = base_layout(
        title=f"{data_type} data | n={n} | Parametric MSE={lr_mse:.2f} | KNN MSE={knn_mse:.2f}",
        xaxis_title="Feature", yaxis_title="Target")
    with out:
        clear_output(wait=True)
        fig = go.Figure(data=traces, layout=layout)
        display(go.FigureWidget(fig))

def on_change(change):
    render(n_slider.value, "violated" in assumption_toggle.value.lower())
n_slider.observe(on_change, names="value")
assumption_toggle.observe(on_change, names="value")
display(VBox([n_slider, assumption_toggle, out]))
render(80, False)

---
## What's happening?

**Parametric models** assume a fixed functional form. Linear regression assumes the relationship is a straight line. Logistic regression assumes a logistic S-curve. The model fits that assumed shape to the data. All the information is compressed into a fixed number of parameters (coefficients), regardless of how much training data you have.

This is like standard shirt sizing: assumes most people fit into S/M/L/XL, works reasonably well for most people, and requires very little information per customer.

**Nonparametric models** make no assumption about shape. KNN remembers the entire training set and makes predictions based on the neighbors of each new point. More data means more neighbors, better coverage, better predictions.

This is like bespoke tailoring: the more measurements you take (data), the better the fit — but you need many more measurements to do the same job.

| Model type | Makes assumptions? | Data needed | Flexibility | Examples |
|---|---|---|---|---|
| Parametric | Yes — fixed functional form | Small | Low-medium | Linear/Logistic Regression, LDA |
| Nonparametric | No | Large | High | KNN, Kernel SVM, Decision Trees |

---
## Real-world example: Predicting loan defaults

A small community bank has 500 historical loans. A logistic regression (parametric) performs well: with few data points, the assumption of a roughly linear risk relationship is a useful constraint that prevents overfitting.

A large online lender has 5 million loan records. A gradient boosting model (nonparametric) discovers complex nonlinear interactions between features that the logistic regression cannot capture: risk spikes for applicants with both high income and high debt-to-income, but not for either factor alone.

More data justifies more flexibility. Less data demands more assumptions.

---
## Key takeaway

> **Parametric models work best when the assumption holds and data is limited; nonparametric models are more flexible but need more data to avoid memorizing noise.**

---
*Next up: Generalization and the Test Set — the final, definitive answer to "how well does my model actually work?"*